# Try the DecoKiller Module

In [1]:
using Revise
using Latexify

includet("../julia/src/DecoKiller.jl")
includet("../julia/src/utils.jl")

using .DecoKiller

In [2]:
cfg, state, logs = load_configuration("../configs/config.toml")

@info cfg.real_noise.name
@info "Starting state:" state.ρ0

[ Info: amplitude_damping
┌ Info: Starting state:
│   state.ρ0 =
│    2×2 Matrix{ComplexF64}:
│      0.549661+0.0im       -0.342071-0.361278im
└     -0.342071+0.361278im   0.450339+0.0im


In [3]:
for step in 1:cfg.n_timesteps
    @info "Step $step"
    step_recovery!(step, state, cfg, logs)

    @info "Fidelities:" logs.fidelities[end] logs.ref_fidelities[end]
end

[ Info: Step 1


MethodError: MethodError: no method matching discrimin(::Matrix{ComplexF64}, ::Matrix{ComplexF64}, ::Matrix{ComplexF64}, ::Matrix{ComplexF64})
The function `discrimin` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  discrimin(::Any, ::Any, ::Any)
   @ Main.DecoKiller ~/UNIPA/COLLISION_MODELS/shooting-decoherences/julia/src/adaptive_recovery.jl:21


In [41]:
ref_fidelities = zeros(Float64, cfg.n_timesteps)
fidelities = zeros(Float64, cfg.n_timesteps)

M_petz = state.noise_guess.supermap_petz
M_noise = state.noise_guess.supermap_noise

model = CollisionModel(state.M_total, cfg.sigma)

display(latexify(state.M_total))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.5676676416183064+0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.4323323583816936+0.0\mathit{i} \\
0.0\mathit{i} & 0.5676676416183064+0.0\mathit{i} & 0.4323323583816936+0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.4323323583816936+0.0\mathit{i} & 0.5676676416183064+0.0\mathit{i} & 0.0\mathit{i} \\
0.4323323583816936+0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.5676676416183064+0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

In [42]:
# 1. Apply noise (update rho1 and create intermediate rho2_)
state.ρ_free = apply_channel(cfg.real_noise.kraus, state.ρ_free, cfg.n_qubits)
ρ_rec_ = apply_channel(cfg.real_noise.kraus, state.ρ_rec, cfg.n_qubits)

# 2. Recovery
# Update rho2 in the state struct
state.ρ_rec, η = apply_collision(model, ρ_rec_)

# 3. Logging
fid_initial = fidelity(state.ρ0, state.ρ_rec)
fid_track = fidelity(state.ρ0, state.ρ_free)

@info "Fidelity wrt initial state: $fid_initial"
@info "Reference fidelity: $fid_track"

[ Info: Fidelity wrt initial state: 0.9999999998
[ Info: Reference fidelity: 0.7700204120020612


In [43]:
state.noise_options[2].kraus

2-element Vector{Matrix{ComplexF64}}:
 [1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.6065306597126334 + 0.0im]
 [0.0 + 0.0im 0.7950600976206501 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im]

In [ ]:
# 4. Measure Ancilla and Update Guess
etas = []
for option in state.noise_options
    update_noise_history!(option)
    _model = CollisionModel(option.supermap, state.ρ0)
    _, _η = apply_collision(_model, ρ_rec_)
    push!(etas, _η)
end


In [ ]:
etas[1]

4×4 Matrix{ComplexF64}:
     0.129589+3.46945e-18im  …  -2.52821e-12-1.85454e-11im
  -0.00771959+0.0561605im        -1.46451e-9-1.65189e-12im
  -1.37795e-9-3.84435e-10im       -0.0422621+0.307459im
 -2.52823e-12+1.85453e-11im         0.709944+0.0im

In [ ]:

povm = measure_ancilla(η, ρ_rec_, state.noise_options, cfg.sigma, cfg.rng)
@info povm

# Update the remaining state variables
state.noise_guess, state.c1, state.c2 = update_noise_guess(
    povm, state.c1, state.c2, state.noise_options)
@info state.noise_guess.name